### Подготовка данных


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from decimal import Decimal

np.random.seed(42)

output_path_sales = Path("../data/sales/sales.csv")
output_path_q1 = Path("../data/sales/q1_sales.csv")
output_path_q2 = Path("../data/sales/q2_sales.csv")
output_path_q3 = Path("../data/sales/q3_sales.csv")
output_path_q4 = Path("../data/sales/q4_sales.csv")

output_path_sales.parent.mkdir(parents=True, exist_ok=True)

n_rows = 2000
dates = pd.date_range("2026-01-01", "2026-12-31", periods=n_rows)
regions = np.random.choice(["Москва", "Санкт-Петербург", "Казань", "Екатеринбург"], size=n_rows)
categories = np.random.choice(["Еда", "Электроника", "Транспорт", "Одежда"], size=n_rows)
sellers = np.random.choice([f"Продавец_{i}" for i in range(1, 21)], size=n_rows)
amounts = np.round(np.random.gamma(shape=2.0, scale=150, size=n_rows), 2)
quantities = np.random.randint(1, 10, size=n_rows)
fractions = np.round(np.random.gamma(shape=2.0, scale=0.03, size=n_rows), 2)

sales_df = pd.DataFrame({
    "event_dt": dates,
    "region": regions,
    "category": categories,
    "seller": sellers,
    "amount": amounts,
    "fraction": fractions,
    "quantity": quantities,
})

sales_df["amount"] = sales_df["amount"].apply(lambda x: Decimal(str(x)))
sales_df["fraction"] = sales_df["fraction"].apply(lambda x: Decimal(str(x)))

# Добавим пустоты
missing_mask = np.random.rand(n_rows) < 0.03
sales_df.loc[missing_mask, "amount"] = np.nan
sales_df.to_csv(output_path_sales, index=False)

# Разобьём на кварталы для примеров с concat
sales_df["event_dt"] = pd.to_datetime(sales_df["event_dt"])
sales_df[sales_df["event_dt"].dt.quarter == 1].to_csv(output_path_q1, index=False)
sales_df[sales_df["event_dt"].dt.quarter == 2].to_csv(output_path_q2, index=False)
sales_df[sales_df["event_dt"].dt.quarter == 3].to_csv(output_path_q3, index=False)
sales_df[sales_df["event_dt"].dt.quarter == 4].to_csv(output_path_q4, index=False)

sales_df.head()

In [ ]:
sales_df.dtypes

### Базовый `groupby`


In [70]:
pd.set_option("display.float_format", "{:.15f}".format)
pd.set_option("display.width", 200)

In [71]:
def to_decimal_zero(x):
    try:
        return Decimal(str(x).strip())
    except Exception:
        return Decimal("0")

In [ ]:
# Decimal
import pandas as pd
import numpy as np

df = pd.read_csv(output_path_sales, converters={"amount": to_decimal_zero, "fraction": to_decimal_zero})

by_region = df.groupby("region")["amount"].sum()
by_region_fraction = df.groupby("region")["fraction"].sum()
print(by_region)
print(by_region_fraction)

In [ ]:
# Float64
import pandas as pd
import numpy as np

df = pd.read_csv(output_path_sales)

by_region = df.groupby("region")["amount"].sum()
by_region_fraction = df.groupby("region")["fraction"].sum()
print(by_region)
print(by_region_fraction)

In [74]:
n = 1_000_000
df_float = pd.DataFrame({
    "a": [10_000_000_000_000_000.0] * n,
    "b": [1.0] * n,
    "c": [10_000_000_000_000_000.0] * n
})

df_float["result"] = (
    df_float["a"] + df_float["b"] - df_float["c"]
)

df_decimal = pd.DataFrame({
    "a": [Decimal("10000000000000000")] * n,
    "b": [Decimal("1")] * n,
    "c": [Decimal("10000000000000000")] * n
})

df_decimal["result"] = (
    df_decimal["a"] + df_decimal["b"] - df_decimal["c"]
)

In [ ]:
print(df_float["result"].sum())
print(df_decimal["result"].sum())

In [ ]:
# Несколько агрегаций
region_stats = df.groupby("region")["amount"].agg(["mean", "sum", "count", "std"])
print(region_stats)

In [ ]:
# Группировка по нескольким столбцам
by_region_category = df.groupby(["region", "category"])["amount"].sum()
print(by_region_category)
print(by_region_category.unstack())

In [78]:
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.width", 200)

### Именованные агрегации

In [ ]:
# Синтаксис: new_name=("column", "function")
summary = df.groupby("region").agg(
    total_revenue=("amount", "sum"),
    avg_check=("amount", "mean"),
    deal_count=("amount", "count"),
    max_check=("amount", "max"),
    unique_sellers=("seller", "nunique"),
)
print(summary)

In [ ]:
# udf в agg
iqr_summary = df.groupby("region").agg(
    iqr=("amount", lambda x: x.quantile(0.75) - x.quantile(0.25)),
    top_quartile_mean=("amount", lambda x: x[x > x.quantile(0.75)].mean()),
)
print(iqr_summary)

### `transform`


In [ ]:
# Добавить средний чек по региону как новый столбец (не сворачивая строки)
df["region_avg_check"] = df.groupby("region")["amount"].transform("mean")

# Доля от суммы региона
df["share_of_region"] = df["amount"] / df.groupby("region")["amount"].transform("sum")

# Z-score внутри каждой группы
df["z_score"] = df.groupby("region")["amount"].transform(
    lambda x: (x - x.mean()) / x.std()
)

df[["region", "amount", "region_avg_check", "share_of_region", "z_score"]].head()


### `pivot_table` и `melt`


In [ ]:
# pivot_table - сводная таблица (как в Excel)
pivot = pd.pivot_table(
    df,
    values="amount",
    index="region",
    columns="category",
    aggfunc="sum",
    fill_value=0,
    margins=True,    
    margins_name="result",
)
print(pivot)


In [ ]:
# melt - обратная операция: "широкая" таблица -> "длинная"
wide_df = pd.DataFrame({
    "region":       ["Екатеринбург", "Москва", "Казань"],
    "Еда":          [100, 200, 150],
    "Электроника":  [300, 250, 400],
    "Транспорт":    [50, 80, 60],
})

long_df = wide_df.melt(
    id_vars=["region"],
    var_name="категория",
    value_name="выручка",
)
print(long_df)


### `pd.concat` - конкатенация датафреймов

In [ ]:
# По строкам (вертикально) — одинаковые столбцы
quarter1 = pd.read_csv(output_path_q1)
quarter2 = pd.read_csv(output_path_q2)
quarter3 = pd.read_csv(output_path_q3)
quarter4 = pd.read_csv(output_path_q4)

full_year_df = pd.concat([quarter1, quarter2, quarter3, quarter4], ignore_index=True)
print(full_year_df.shape)
full_year_df.head()

In [ ]:
# По столбцам (горизонтально)
features_df = pd.DataFrame({"a": [1, 2, 3]})
targets_df = pd.DataFrame({"target": [0, 1, 0]})

combined_df = pd.concat([features_df, targets_df], axis=1)
print(combined_df)


### `merge` - аналог SQL JOIN

In [ ]:
transactions = pd.DataFrame({
    "id":         [1, 2, 3, 4, 5],
    "client_id":  [101, 102, 101, 103, 102],
    "amount":      [250, 1200, 89, 500, 300],
})

customers = pd.DataFrame({
    "client_id": [101, 102, 103, 104],
    "имя":       ["Иванов", "Петрова", "Сидоров", "Козлов"],
    "сегмент":   ["VIP", "Стандарт", "VIP", "Стандарт"],
})

# INNER JOIN - только совпадающие записи
inner_df = pd.merge(transactions, customers, on="client_id", how="inner")
print(inner_df)


In [ ]:
# LEFT JOIN - все транзакции, даже без клиента
left_df = pd.merge(transactions, customers, on="client_id", how="left")
print(left_df)

In [ ]:
# Разные имена ключевых столбцов
customers_alt = customers.rename(columns={"client_id": "id_клиента"})
alt_df = pd.merge(transactions, customers_alt,
                   left_on="client_id", right_on="id_клиента")
print(alt_df)

In [ ]:
# Проверка объединения
validated_df = pd.merge(transactions, customers, on="client_id",
                         how="left", validate="many_to_one")
# validate поднимет исключение, если в "один" есть дубликаты - это сигнал о проблеме в данных
print(validated_df)

In [ ]:
# Проверить пропуски после LEFT JOIN
print(left_df[left_df["имя"].isna()])  # транзакции без клиента

### `join` - упрощённый merge по индексу

In [ ]:
revenue_df = pd.DataFrame({"выручка": [100, 200]}, index=["A", "B"])
expenses_df = pd.DataFrame({"расходы": [80, 150]}, index=["A", "B"])

joined_df = revenue_df.join(expenses_df)
print(joined_df)

### Работа с пропусками

In [ ]:
df = pd.read_csv(output_path_sales)

# Обнаружение пропусков
print(df.isna().sum())          # сколько пропусков в каждом столбце
print(df.isna().mean() * 100)   # доля пропусков в %


In [ ]:
# Удаление строк с пропусками
df_dropna_all = df.dropna()                          # удалить все строки с хоть одним NaN
df_dropna_subset = df.dropna(subset=["amount", "event_dt"])  # только если NaN в ключевых столбцах
df_dropna_thresh = df.dropna(thresh=5)                # оставить строки с >= 5 непустыми значениями

print(df_dropna_all.shape)


In [ ]:
# Заполнение пропусков
df["amount"] = df["amount"].fillna(df["amount"].median())    # медиана
df["region"] = df["region"].fillna("Неизвестно")             # константа
df["amount"] = df["amount"].ffill()                          # предыдущее значение
df["amount"] = df["amount"].bfill()                          # следующее значение

df.isna().sum()

### Категориальный dtype


In [ ]:
# Экономит память при небольшом числе уникальных значений
print(df["category"].dtype)         # object (строки)
df["category"] = df["category"].astype("category")
print(df["category"].dtype)         # category


In [ ]:
print(df["category"].cat.categories)    # список уникальных категорий
print(df["category"].cat.codes.head())  # числовые коды

In [ ]:
# Экономия памяти
bytes_before = df["category"].astype("object").nbytes
df["category"] = df["category"].astype("category")
bytes_after = df["category"].nbytes
print(f"До: {bytes_before} байт, после: {bytes_after} байт")


### Работа с датами и временем


In [ ]:
df["event_dt"] = pd.to_datetime(df["event_dt"])

# Аксессор .dt - извлечение компонентов
df["year"]         = df["event_dt"].dt.year
df["month"]        = df["event_dt"].dt.month
df["weekday"]      = df["event_dt"].dt.day_name()
df["quarter"]      = df["event_dt"].dt.quarter
df["week_of_year"] = df["event_dt"].dt.isocalendar().week

df[["event_dt", "year", "month", "weekday", "quarter", "week_of_year"]].head()

In [ ]:
# Арифметика с датами
df["days_since_start"] = (df["event_dt"] - df["event_dt"].min()).dt.days
df[["event_dt", "days_since_start"]].head()

In [ ]:
# resample временных рядов
df_ts = df.set_index("event_dt")

# agg по периодам
monthly = df_ts["amount"].resample("ME").sum()      # ежемесячно
quarterly = df_ts["amount"].resample("QE").mean()   # поквартально
weekly = df_ts["amount"].resample("W").agg(["sum", "count"])

print(monthly.head())

In [ ]:
# Скользящее среднее
df_ts["rolling_mean_7d"] = df_ts["amount"].rolling(window=7).mean()
df_ts["rolling_mean_30d"] = df_ts["amount"].rolling(window=30).mean()

df_ts[["amount", "rolling_mean_7d", "rolling_mean_30d"]].head(10)

### `.describe()` и первичный осмотр

In [ ]:
df = pd.read_csv(output_path_sales, parse_dates=["event_dt"])

# Числовые столбцы
print(df.describe())
# count, mean, std, min, 25%, 50%, 75%, max

In [ ]:
# Категориальные столбцы
print(df.describe(include=["object", "str"]))
# count, unique, top (самое частое), freq

In [ ]:
# Все столбцы
df.describe(include="all")

### `value_counts` и распределения


In [ ]:
# Частотное распределение
print(df["category"].value_counts())
print(df["category"].value_counts(normalize=True) * 100)  # в %

In [ ]:
# Разбивка по диапазонам
df["check_size"] = pd.cut(
    df["amount"],
    bins=[0, 100, 500, 1000, float("inf")],
    labels=["малый", "средний", "крупный", "очень крупный"]
)
print(df["check_size"].value_counts())


### Обнаружение выбросов


In [ ]:
# Метод IQR
q1_value = df["amount"].quantile(0.25)
q3_value = df["amount"].quantile(0.75)
iqr_value = q3_value - q1_value

lower_bound = q1_value - 1.5 * iqr_value
upper_bound = q3_value + 1.5 * iqr_value

outliers_iqr = df[(df["amount"] < lower_bound) | (df["amount"] > upper_bound)]
print(f"Выбросов: {len(outliers_iqr)} ({len(outliers_iqr) / len(df) * 100:.1f}%)")

In [ ]:
# Метод Z-score
df["z_score"] = (df["amount"] - df["amount"].mean()) / df["amount"].std()
outliers_z = df[df["z_score"].abs() > 3]
print(f"Выбросов по Z-score: {len(outliers_z)}")